In [1]:
import ast
import hashlib
import heapq
import json
import os
import re
from collections import Counter, defaultdict

import networkx as nx
import numpy as np
import pandas as pd
from git import Repo


# Pre-Process Python File


In [2]:
def py_to_ast(py_code):
    try:
        return ast.parse(py_code)
    except SyntaxError as error:
        raise ValueError(f"Syntax error in the provided Python code: {error}") from error

# Global Static Variables


In [3]:
LEAF = (ast.expr_context, ast.operator, ast.boolop, ast.unaryop, ast.cmpop)
FUNC = (ast.FunctionDef, ast.AsyncFunctionDef)
SCOPES = FUNC + (ast.ClassDef, ast.Lambda, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

BINDERS = {getattr(ast, n): f for n, f in [("ExceptHandler", "name"), ("MatchAs", "name"), ("MatchStar", "name"), ("MatchMapping", "rest")] if hasattr(ast, n)}

CONTROL = {getattr(ast, n): (c, b) for n, c, b in [
    ("If",            ("test",),            ("body", "orelse")),
    ("While",         ("test",),            ("body", "orelse")),
    ("For",           ("iter",),            ("body", "orelse")),
    ("AsyncFor",      ("iter",),            ("body", "orelse")),
    ("With",          ("items",),           ("body",)),
    ("AsyncWith",     ("items",),           ("body",)),
    ("Try",           (),                   ("body", "handlers", "orelse", "finalbody")),
    ("TryStar",       (),                   ("body", "handlers", "orelse", "finalbody")),
    ("ExceptHandler", ("type",),            ("body",)),
    ("Match",         ("subject",),         ("cases",)),
    ("match_case",    ("pattern", "guard"), ("body",)),
] if hasattr(ast, n)}

EDGE_PRIORITY = {
    "call": 0,
    "data_dep": 1,
    "import": 2,
    "control": 3,
    "arg_bind": 4,
    "return_flow": 5,
    "class_method": 6,
    "child": 7,
}

NODE_TYPES = tuple(name for name in dir(ast) if isinstance(getattr(ast, name), type) and issubclass(getattr(ast, name), ast.AST))
NODE_TYPE_TO_ID = {name: index for index, name in enumerate(NODE_TYPES)}

EDGE_TYPES = (
    "child",
    "call",
    "data_dep",
    "import",
    "control",
    "arg_bind",
    "return_flow",
    "class_method",
)

EDGE_TYPE_TO_ID = {name: index for index, name in enumerate(EDGE_TYPES)}

AST_TYPES = tuple(name for name in sorted(dir(ast)) if isinstance(getattr(ast, name), type) and issubclass(getattr(ast, name), ast.AST))
AST_TYPE_TO_ID = {name: index for index, name in enumerate(AST_TYPES)}

CHANGE_STATUS_TO_ID = {"unchanged": 0, "modified": 1, "added": 2, "deleted": 3}

TOKEN_BUCKETS = 64

# Graph Builder


In [4]:
class Graph:
    def __init__(self):
        self.graph = nx.MultiDiGraph()

    def add_node(self, node_idx, node_type, start_lineno = None, end_lineno = None, attrs = None):
        self.graph.add_node(node_idx, type = node_type, start_lineno = start_lineno, end_lineno = end_lineno, attrs = attrs or {})

    def add_edge(self, from_node, to_node, edge_type, **attrs):
        self.graph.add_edge(from_node, to_node, type = edge_type, **attrs)

class Scope:
    def __init__(self, node, parent, path = None):
        self.node, self.parent, self.path = node, parent, path
        self.defs = defaultdict(list)

class Builder:
    def __init__(self):
        self.graph = Graph()
        self._ids = {}
        self.scope_of = {}
        self.own_scope = {}
        self._defpos = {}
        self.module_scope = {}

    @staticmethod
    def _module_name(path):
        path = path.replace("\\", "/")
        if path.endswith(".py"):
            path = path[:-3]
        parts = path.split("/")
        if parts[-1] == "__init__":
            parts = parts[:-1]
        return ".".join(parts)

    @staticmethod
    def _children(node):
        for field, value in ast.iter_fields(node):
            items = value if isinstance(value, list) else [value]
            for order, child in enumerate(items):
                if isinstance(child, ast.AST) and not isinstance(child, LEAF):
                    yield field, order, child

    @staticmethod
    def _attrs(node):
        attrs = {}
        for field, value in ast.iter_fields(node):
            if isinstance(value, LEAF):
                attrs[field] = type(value).__name__
            elif isinstance(value, list):
                if value and all(isinstance(v, LEAF) for v in value):
                    attrs[field] = [type(v).__name__ for v in value]
                elif value and all(isinstance(v, str) for v in value):
                    attrs[field] = list(value)
            elif value is not None and not isinstance(value, ast.AST):
                attrs[field] = value
        return attrs

    @staticmethod
    def _register(node, scope):
        if isinstance(node, FUNC + (ast.ClassDef,)):
            scope.defs[node.name].append(node)
        elif isinstance(node, ast.arg):
            scope.defs[node.arg].append(node)
        elif isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store):
            scope.defs[node.id].append(node)
        elif isinstance(node, ast.alias):
            name = node.asname or node.name.split(".")[0]
            if name != "*":
                scope.defs[name].append(node)
        elif type(node) in BINDERS:
            name = getattr(node, BINDERS[type(node)])
            if name:
                scope.defs[name].append(node)

    @staticmethod
    def _inside(node, field):
        if isinstance(node, FUNC):
            return field in ("args", "body")
        if isinstance(node, ast.ClassDef):
            return field == "body"
        return True

    @staticmethod
    def _lookup(scope, name):
        s, start = scope, True
        while s is not None:
            if name in s.defs and (start or not isinstance(s.node, ast.ClassDef)):
                return s
            s, start = s.parent, False
        return None

    def build(self, trees):
        if isinstance(trees, ast.AST):
            trees = {"module.py": trees}

        for path, tree in trees.items():
            stack = [(tree, None, None, None)]
            while stack:
                node, parent, field, order = stack.pop()
                idx = self._ids.setdefault(node, len(self._ids))
                self.graph.add_node(idx, type(node).__name__, start_lineno = getattr(node, 'lineno', None), end_lineno = getattr(node, 'end_lineno', None), attrs = self._attrs(node))
                if parent is not None:
                    self.graph.add_edge(parent, idx, "child", field = field, order = order)
                for field, order, child in reversed(list(self._children(node))):
                    stack.append((child, idx, field, order))
            self.graph.graph.nodes[self._ids[tree]]["attrs"]["path"] = path

        all_nodes = [n for t in trees.values() for n in ast.walk(t)]

        for n in all_nodes:
            if isinstance(n, (ast.Assign, ast.AnnAssign, ast.AugAssign, ast.NamedExpr)):
                targets = n.targets if isinstance(n, ast.Assign) else [n.target]
                for t in targets:
                    for m in ast.walk(t):
                        if isinstance(m, ast.Name) and isinstance(m.ctx, ast.Store):
                            self._defpos[m] = (n.end_lineno, n.end_col_offset)

        for path, tree in trees.items():
            root = self.module_scope[path] = Scope(tree, None, path)
            stack = [(tree, root)]
            while stack:
                node, scope = stack.pop()
                self.scope_of[node] = scope
                self._register(node, scope)
                inner = scope
                if isinstance(node, SCOPES):
                    inner = self.own_scope[node] = Scope(node, scope)
                for field, _, child in self._children(node):
                    if isinstance(node, ast.arguments) and field in ("defaults", "kw_defaults"):
                        cs = scope.parent
                    elif isinstance(node, SCOPES) and not self._inside(node, field):
                        cs = scope
                    else:
                        cs = inner
                    stack.append((child, cs))

        modules = {self._module_name(p): r for p, r in self.module_scope.items()}
        info, alias_defs, alias_mods = {}, {}, {}

        def module_of(scope):
            while scope.parent is not None:
                scope = scope.parent
            return scope.path

        for imp in all_nodes:
            if isinstance(imp, (ast.Import, ast.ImportFrom)):
                for al in imp.names:
                    info[al] = (imp, module_of(self.scope_of[imp]))

        def find_module(name):
            if not name:
                return None
            if name in modules:
                return modules[name]
            cand = [m for m in modules if m.endswith("." + name)]
            return modules[cand[0]] if len(cand) == 1 else None

        def absolute_module(imp, path):
            if imp.level == 0:
                return imp.module or ""
            parts = self._module_name(path).split(".")
            if not path.endswith("__init__.py"):
                parts = parts[:-1]
            parts = parts[:max(0, len(parts) - (imp.level - 1))]
            return ".".join(parts + (imp.module.split(".") if imp.module else []))

        def resolve_alias(al, seen=()):
            if al in alias_defs:
                return
            imp, path = info[al]
            defs, mods = [], []
            if isinstance(imp, ast.Import):
                m = find_module(al.name if al.asname else al.name.split(".")[0])
                if m:
                    mods.append(m)
            elif al.name != "*":
                base = absolute_module(imp, path)
                m = find_module(base)
                for d in (m.defs.get(al.name, []) if m else []):
                    if isinstance(d, ast.alias) and d in info and d is not al and d not in seen:
                        resolve_alias(d, seen + (al,))
                        defs += alias_defs.get(d, [])
                        mods += alias_mods.get(d, [])
                    else:
                        defs.append(d)
                if not defs and not mods:
                    sub = find_module(f"{base}.{al.name}" if base else al.name)
                    if sub:
                        mods.append(sub)
            alias_defs[al], alias_mods[al] = defs, mods

        for al in info:
            resolve_alias(al)
        for al in info:
            for d in alias_defs[al]:
                self.graph.add_edge(self._ids[al], self._ids[d], "import")
            for m in alias_mods[al]:
                self.graph.add_edge(self._ids[al], self._ids[m.node], "import")

        def expand(ds):
            out = []
            for d in ds:
                out += alias_defs[d] if isinstance(d, ast.alias) else [d]
            return out

        def defs_of(scope, name, raw=False):
            s = self._lookup(scope, name)
            ds = s.defs[name] if s else []
            return ds if raw else expand(ds)

        def class_scopes(scope, name):
            s = scope
            while s is not None:
                if isinstance(s.node, FUNC) and s.parent and isinstance(s.parent.node, ast.ClassDef):
                    a = s.node.args
                    params = a.posonlyargs + a.args
                    if params and params[0].arg == name:
                        return [s.parent], True
                    break
                s = s.parent
            out = [self.own_scope[d] for d in defs_of(scope, name) if isinstance(d, ast.ClassDef)]
            for d in defs_of(scope, name, raw=True):
                if isinstance(d, ast.alias):
                    out += alias_mods[d]
            return out, False

        def pos(d):
            return self._defpos.get(d) or (getattr(d, "lineno", 0), getattr(d, "col_offset", 0))

        def add_target(found, d, offset):
            if isinstance(d, FUNC):
                found.append((d, offset))
            elif isinstance(d, ast.ClassDef):
                found.append((d, 0))
                for m in self.own_scope[d].defs.get("__init__", []):
                    if isinstance(m, FUNC):
                        found.append((m, 1))

        for cls in all_nodes:
            if isinstance(cls, ast.ClassDef):
                for stmt in cls.body:
                    if isinstance(stmt, FUNC):
                        self.graph.add_edge(self._ids[cls], self._ids[stmt], "class_method")

        calls = []
        for call in all_nodes:
            if not isinstance(call, ast.Call):
                continue
            f, scope = call.func, self.scope_of[call]
            found = []
            if isinstance(f, ast.Name):
                for d in defs_of(scope, f.id):
                    add_target(found, d, 0)
            elif isinstance(f, ast.Attribute) and isinstance(f.value, ast.Name):
                scopes, bound = class_scopes(scope, f.value.id)
                for cs in scopes:
                    for d in expand(cs.defs.get(f.attr, [])):
                        if isinstance(d, FUNC) and isinstance(cs.node, ast.ClassDef):
                            decos = {x.id for x in d.decorator_list if isinstance(x, ast.Name)}
                            offset = 0 if "staticmethod" in decos else 1 if (bound or "classmethod" in decos) else 0
                            add_target(found, d, offset)
                        else:
                            add_target(found, d, 0)
            for t, offset in found:
                self.graph.add_edge(self._ids[call], self._ids[t], "call")
                calls.append((call, t, offset))

        callee = {c.func for c in all_nodes if isinstance(c, ast.Call)}
        aug = {n.target for n in all_nodes if isinstance(n, ast.AugAssign) and isinstance(n.target, ast.Name)}
        for use in all_nodes:
            if not isinstance(use, ast.Name) or use in callee:
                continue
            if not (isinstance(use.ctx, ast.Load) or use in aug):
                continue
            scope = self.scope_of[use]
            found = self._lookup(scope, use.id)
            if not found:
                continue
            defs = found.defs[use.id]
            if found is scope:
                before = [d for d in defs if pos(d) < (use.lineno, use.col_offset)]
                defs = before or defs
            for d in defs:
                if d is not use:
                    self.graph.add_edge(self._ids[d], self._ids[use], "data_dep")

        returns = {}

        def returns_of(fn):
            if fn not in returns:
                out, st = [], list(fn.body)
                while st:
                    n = st.pop()
                    if isinstance(n, ast.Return):
                        out.append(n)
                    if not isinstance(n, FUNC + (ast.ClassDef, ast.Lambda)):
                        st.extend(ast.iter_child_nodes(n))
                returns[fn] = out
            return returns[fn]

        for call, t, offset in calls:
            if isinstance(t, ast.ClassDef):
                continue
            a = t.args
            params = a.posonlyargs + a.args
            by_name = {p.arg: p for p in params + a.kwonlyargs}
            for i, arg in enumerate(call.args):
                if not isinstance(arg, ast.Starred) and i + offset < len(params):
                    self.graph.add_edge(self._ids[arg], self._ids[params[i + offset]], "arg_bind")
            for kw in call.keywords:
                if kw.arg in by_name:
                    self.graph.add_edge(self._ids[kw.value], self._ids[by_name[kw.arg]], "arg_bind")
            for r in returns_of(t):
                if r.value is not None:
                    self.graph.add_edge(self._ids[r], self._ids[call], "return_flow")

        for n in all_nodes:
            rule = CONTROL.get(type(n))
            if rule is None:
                continue
            ctrl_fields, block_fields = rule
            sources = [c for f, _, c in self._children(n) if f in ctrl_fields] or [n]
            for f, _, child in self._children(n):
                if f in block_fields:
                    for s in sources:
                        self.graph.add_edge(self._ids[s], self._ids[child], "control", field = f)

        return self.graph

# Graph Consistency Checks


In [5]:
DEF_SRC = FUNC + (ast.ClassDef, ast.arg, ast.Name, ast.alias, ast.ExceptHandler) + tuple(BINDERS)
RULES = {
    "call":         ((ast.Call,),     FUNC + (ast.ClassDef,)),
    "arg_bind":     ((ast.expr,),     (ast.arg,)),
    "return_flow":  ((ast.Return,),   (ast.Call,)),
    "class_method": ((ast.ClassDef,), FUNC),
    "import":       ((ast.alias,),    FUNC + (ast.ClassDef, ast.Name, ast.alias, ast.Module)),
    "data_dep":     (DEF_SRC,         (ast.Name,)),
}

def check_graph(builder, trees):
    g = builder.graph.graph
    id2 = {i: n for n, i in builder._ids.items()}
    problems, notes = [], []

    exp = sum(1 for t in trees.values() for n in ast.walk(t) if not isinstance(n, LEAF))
    child = sum(1 for *_, d in g.edges(data=True) if d["type"] == "child")
    if g.number_of_nodes() != exp:
        problems.append(f"jumlah node {g.number_of_nodes()} != {exp}")
    if child != exp - len(trees):
        problems.append(f"edge child {child} != {exp - len(trees)}")

    ghost = [i for i, d in g.nodes(data=True) if "type" not in d]
    if ghost:
        problems.append(f"{len(ghost)} node hantu, contoh id {ghost[:5]}")

    bad = Counter()
    for u, v, d in g.edges(data=True):
        rule = RULES.get(d["type"])
        if rule and not (isinstance(id2[u], rule[0]) and isinstance(id2[v], rule[1])):
            bad[d["type"]] += 1
    if bad:
        problems.append(f"tipe ujung edge tidak sesuai: {dict(bad)}")

    tops = {Builder._module_name(p).split(".")[0] for p in trees}
    resolved = {u for u, v, d in g.edges(data=True) if d["type"] == "import"}
    unresolved = []
    for path, tree in trees.items():
        for imp in ast.walk(tree):
            if isinstance(imp, ast.ImportFrom):
                internal = imp.level > 0 or (imp.module or "").split(".")[0] in tops
            elif isinstance(imp, ast.Import):
                internal = any(a.name.split(".")[0] in tops for a in imp.names)
            else:
                continue
            if internal:
                for al in imp.names:
                    if al.name != "*" and builder._ids[al] not in resolved:
                        unresolved.append((path, imp.lineno, ast.unparse(imp)))
    if unresolved:
        notes.append(f"{len(unresolved)} import internal tidak ter-resolve")
        notes += [f"   {p}:{ln}  {s}" for p, ln, s in unresolved[:8]]

    ctrl_stmts = sum(isinstance(n, tuple(CONTROL)) for t in trees.values() for n in ast.walk(t))
    print(f"node={g.number_of_nodes()} (diharapkan {exp}) | file={len(trees)} | statement kontrol={ctrl_stmts}")
    print("edge:", dict(Counter(d['type'] for *_, d in g.edges(data=True))))
    print("MASALAH:" if problems else "tidak ada masalah konsistensi")
    for p in problems: print("  -", p)
    for n in notes: print("  *", n)
    return not problems



# Training Input Schema

In [6]:
def _stable_hash(value, buckets = 64):
    digest = hashlib.blake2b(str(value).encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, "big") % buckets

def _node_tokens(attrs):
    values = [attrs.get("type", "")]
    ast_attrs = attrs.get("attrs", {})
    for key in sorted(ast_attrs):
        value = ast_attrs[key]
        if isinstance(value, (str, int, float, bool)):
            values.append(f"{key}={value}")
        elif isinstance(value, list):
            values.extend(f"{key}={item}" for item in value)
    return values

def diff_metadata(diff_items):
    result = {
        "old_ranges": {},
        "new_ranges": {},
        "edits": [],
        "changed_files": [],
    }
    header = re.compile(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@")
    for item in diff_items:
        old_path = item.a_path
        new_path = item.b_path
        path = new_path or old_path
        if not path or not path.endswith(".py"):
            continue
        result["changed_files"].append(path)
        old_ranges, new_ranges = [], []
        old_line = new_line = None
        removed, added = [], []
        removed_start = added_start = None

        def flush():
            nonlocal removed, added, removed_start, added_start
            if removed or added:
                if removed and added:
                    operation = "Update"
                elif removed:
                    operation = "Delete"
                else:
                    operation = "Insert"
                result["edits"].append(
                    {
                        "operation": operation,
                        "path": path,
                        "old_start": removed_start,
                        "new_start": added_start,
                        "old_text": "\n".join(removed),
                        "new_text": "\n".join(added),
                    }
                )
            removed = []
            added = []
            removed_start = added_start = None

        for line in item.diff.decode("utf-8", errors = "replace").splitlines():
            match = header.match(line)
            if match:
                flush()
                old_line = int(match.group(1))
                new_line = int(match.group(3))
                continue
            if old_line is None or line.startswith("\\"):
                continue
            if line.startswith("-") and not line.startswith("---"):
                if removed_start is None:
                    removed_start = old_line
                removed.append(line[1:])
                old_ranges.append((old_line, old_line))
                old_line += 1
            elif line.startswith("+") and not line.startswith("+++"):
                if added_start is None:
                    added_start = new_line
                added.append(line[1:])
                new_ranges.append((new_line, new_line))
                new_line += 1
            else:
                flush()
                old_line += 1
                new_line += 1
        flush()
        if old_ranges:
            result["old_ranges"][old_path or path] = old_ranges
        if new_ranges:
            result["new_ranges"][new_path or path] = new_ranges

    for key in ("old_ranges", "new_ranges"):
        for path, ranges in result[key].items():
            ranges.sort()
            merged = []
            for start, end in ranges:
                if merged and start <= merged[-1][1] + 1:
                    merged[-1] = (merged[-1][0], max(merged[-1][1], end))
                else:
                    merged.append((start, end))
            result[key][path] = merged
    return result

def _range_overlaps(attrs, ranges):
    start = attrs.get("start_lineno")
    end = attrs.get("end_lineno")
    return (
        start is not None
        and end is not None
        and any(start <= right and left <= end for left, right in ranges)
    )

def prepare_pruned_graph(builder, trees, changed_ranges, max_nodes):
    graph = builder.graph.graph
    metadata = {}
    anchors = set()
    for path, tree in trees.items():
        for node in ast.walk(tree):
            if node not in builder._ids:
                continue
            node_id = builder._ids[node]
            info = {
                "path": path,
                "ast_node": node,
                "start_lineno": getattr(node, "lineno", None),
                "end_lineno": getattr(node, "end_lineno", None),
            }
            metadata[node_id] = info
            if _range_overlaps(
                info,
                changed_ranges.get(path, ()),
            ):
                anchors.add(node_id)
    if not anchors:
        raise ValueError("Tidak ditemukan node AST pada changed_ranges")
    if len(anchors) > max_nodes:
        raise ValueError("Jumlah change anchor melebihi max_nodes")

    priority = {
        "call": 0,
        "data_dep": 1,
        "import": 2,
        "control": 3,
        "arg_bind": 4,
        "return_flow": 5,
        "class_method": 6,
        "child": 7,
    }
    best = {node_id: (0, -1, node_id) for node_id in anchors}
    queue = list(best.values())
    heapq.heapify(queue)
    while queue:
        distance, rank, node_id = heapq.heappop(queue)
        if best.get(node_id) != (distance, rank, node_id):
            continue
        neighbors = [
            (source, attrs.get("type"))
            for source, _, attrs in graph.in_edges(node_id, data=True)
        ]
        neighbors.extend(
            (target, attrs.get("type"))
            for _, target, attrs in graph.out_edges(node_id, data=True)
        )
        for neighbor, edge_type in neighbors:
            candidate = (
                distance + 1,
                priority.get(edge_type, len(priority)),
                neighbor,
            )
            if neighbor not in best or candidate < best[neighbor]:
                best[neighbor] = candidate
                heapq.heappush(queue, candidate)
    selected = sorted(best, key=best.get)[:max_nodes]
    result = graph.subgraph(selected).copy()
    result = nx.convert_node_labels_to_integers(
        result, first_label=0, ordering="sorted", label_attribute="original_id"
    )
    result.graph["max_nodes"] = max_nodes
    result.graph["anchor_original_ids"] = sorted(anchors)
    anchors = set(result.graph.get("anchor_original_ids", ()))
    for node_id, attrs in result.nodes(data=True):
        original_id = attrs.get("original_id", node_id)
        info = metadata.get(original_id, {})
        source_attrs = dict(attrs)
        source_attrs["path"] = info.get("path")
        source_attrs["start_lineno"] = info.get("start_lineno")
        source_attrs["end_lineno"] = info.get("end_lineno")
        source_attrs["ast_node"] = info.get("ast_node")
        source_attrs["is_change_anchor"] = int(original_id in anchors)
        source_attrs["is_changed_file"] = int(
            info.get("path") in changed_ranges
        )
        result.nodes[node_id].update(source_attrs)
    return result

class Tensor:
    def __init__(
        self,
        pruned_graph,
        graph_features=None,
        local_change_features=None,
    ):
        self.graph = pruned_graph
        self.graph_features = self._features(graph_features)
        self.local_change_features = self._features(local_change_features)
        self.node_ids = list(pruned_graph.nodes())
        self.node_to_index = {node: i for i, node in enumerate(self.node_ids)}
        self.x = self._node_features()
        self.edge_index, self.edge_type = self._edges()

    @staticmethod
    def _features(values):
        if values is None:
            return np.empty(0, dtype=np.float32)
        if isinstance(values, dict):
            values = [values[key] for key in sorted(values)]
        vector = np.asarray(values, dtype=np.float32).reshape(-1)
        if not np.isfinite(vector).all():
            raise ValueError("Fitur mengandung NaN atau infinite")
        return vector

    def _node_features(self):
        type_count = len(AST_TYPES) + 1
        width = type_count + TOKEN_BUCKETS + 7
        output = np.zeros((len(self.node_ids), width), dtype=np.float32)
        line_numbers = [
            value
            for node in self.node_ids
            for value in (
                self.graph.nodes[node].get("start_lineno"),
                self.graph.nodes[node].get("end_lineno"),
            )
            if isinstance(value, (int, float))
        ]
        max_line = max(line_numbers, default=1)
        for row, node in enumerate(self.node_ids):
            attrs = self.graph.nodes[node]
            type_id = AST_TYPE_TO_ID.get(attrs.get("type"), len(AST_TYPES))
            output[row, type_id] = 1.0
            offset = type_count
            for token in _node_tokens(attrs):
                output[row, offset + _stable_hash(token)] = 1.0
            start = attrs.get("start_lineno") or 0
            end = attrs.get("end_lineno") or start
            output[row, offset + TOKEN_BUCKETS] = start / max_line
            output[row, offset + TOKEN_BUCKETS + 1] = end / max_line
            output[row, offset + TOKEN_BUCKETS + 2] = attrs.get(
                "is_change_anchor", 0
            )
            output[row, offset + TOKEN_BUCKETS + 3] = attrs.get(
                "is_changed_file", 0
            )
            output[row, offset + TOKEN_BUCKETS + 4] = int(
                bool(attrs.get("attrs", {}).get("ctx"))
            )
            output[row, offset + TOKEN_BUCKETS + 5] = _stable_hash(
                attrs.get("path", ""), 16
            ) / 16
            output[row, offset + TOKEN_BUCKETS + 6] = int(
                attrs.get("ast_node") is not None
            )
        return output

    def _edges(self):
        edges, types = [], []
        for source, target, attrs in self.graph.edges(data=True):
            edge_type = attrs.get("type")
            if edge_type not in EDGE_TYPE_TO_ID:
                raise ValueError(f"Tipe edge tidak dikenal: {edge_type!r}")
            edges.append(
                [self.node_to_index[source], self.node_to_index[target]]
            )
            types.append(EDGE_TYPE_TO_ID[edge_type])
        index = np.asarray(edges, dtype=np.int64).T if edges else np.empty((2, 0), dtype=np.int64)
        return index, np.asarray(types, dtype=np.int64)

def graph_features(graph):
    counts = Counter(attrs.get("type") for _, _, attrs in graph.edges(data=True))
    values = [
        graph.number_of_nodes(),
        graph.number_of_edges(),
        len(graph.graph.get("anchor_original_ids", ())),
    ]
    values.extend(counts.get(edge_type, 0) for edge_type in EDGE_TYPES)
    return np.asarray(values, dtype=np.float32)

def pair_features(diff_info):
    edits = diff_info["edits"]
    operation_ids = {"Insert": 1, "Delete": 2, "Update": 3}
    counts = Counter(edit["operation"] for edit in edits)
    return np.asarray(
        [
            len(diff_info["changed_files"]),
            len(edits),
            counts.get("Insert", 0),
            counts.get("Delete", 0),
            counts.get("Update", 0),
            sum(len(edit["old_text"].split()) for edit in edits),
            sum(len(edit["new_text"].split()) for edit in edits),
            sum(operation_ids[edit["operation"]] for edit in edits),
        ],
        dtype=np.float32,
    )

def sample_target(value):
    if value is None:
        return None
    if isinstance(value, str):
        value = value.strip().lower()
        if value in {"true", "1", "yes"}:
            return np.asarray([1.0], dtype=np.float32)
        if value in {"false", "0", "no"}:
            return np.asarray([0.0], dtype=np.float32)
    return np.asarray([float(bool(value))], dtype=np.float32)


# Load Dataset


In [7]:
dataset_path = "/home/darel/workspaces/tugas/rpl_ml/datasets/file-9.jsonl"
with open(dataset_path, "r", encoding="utf-8") as file:
    dataset = [json.loads(line) for line in file]
df = pd.DataFrame(dataset)
print(f"Dataset shape: {len(df)}")
display(df.head())


Dataset shape: 66985


,project,commit_sha,parent_sha,file_path,project_url,likely_bug,comodified,in_function,diff,before,after,sstub_pattern,edit_script
0,gajim,2e9c83914b3bf2150d578521538b3b86fe3dedd3,e50b237ff2411bf0bee3240b79619bf5e3d88540,src/common/config.py,https://github.com/sgala/gajim,True,False,False,"@@ -161,7 +161,7 @@ class Config:\n \t\t'show_...","'log_contact_status_changes' : [ opt_bool , Tr...","'log_contact_status_changes' : [ opt_bool , Fa...",CHANGE_BOOLEAN_LITERAL,"[[""Insert"", [""subscript"", 3, 3, 3, 49], [""fals..."
1,gajim,bc6f4289bef834e66a75e36ed2b20625d95b85cd,8c4288cc2c3316a33538348ddf34030a26d502a2,src/vcard.py,https://github.com/sgala/gajim,True,False,True,"@@ -281,7 +281,7 @@ class VcardWindow:\n \t\ts...","tt_text = _ ( ""This contact is interested in y...","tt_text = _ ( ""This contact is interested in y...",CHANGE_STRING_LITERAL,"[[""Update"", [""string:\""This contact is interes..."
2,gajim,90a4fe3936ba28b683fe36737e0f8acaa841bed7,107cd477d03825cca5f097b26b3d968bb042768f,src/gajim.py,https://github.com/sgala/gajim,True,False,True,"@@ -1335,7 +1335,7 @@ class Interface:\n \t\t\...",try : import remote_control self . remote_ctrl...,try : import remote_control self . remote_ctrl...,SINGLE_STMT,"[[""Update"", [""identifier:remote_control"", 3, 1..."
3,gajim,44267b410c1697a316f08fea8b0cc542a7193aa0,1962321a2639ed0aa23ad4bd62056f7db16c98ce,src/filetransfers_window.py,https://github.com/sgala/gajim,True,False,True,"@@ -459,7 +459,7 @@ _('Connection with peer ca...",text += _ ( '(%(filesize_unit)d/s)' ) % { 'fil...,text += _ ( '(%(filesize_unit)s/s)' ) % { 'fil...,CHANGE_STRING_LITERAL,"[[""Update"", [""string:'(%(filesize_unit)d/s)'"",..."
4,gajim,5365c16a7cffd761b64157ff686840988dc05ac1,d48bf500ce97e0d900805ce420710c427dc0a2bd,src/tabbed_chat_window.py,https://github.com/sgala/gajim,True,False,True,"@@ -845,7 +845,7 @@ class TabbedChatWindow(cha...",if jid == self . get_active_jid ( ) and gajim ...,if gajim . interface . systray_enabled : gajim...,SINGLE_STMT,"[[""Move"", [""if_statement"", 3, 3, 4, 62], [""att..."


# Load Repository and Commit Pair


In [8]:
folder_path = "/home/darel/workspaces/tugas/rpl_ml/repositories"
project_name = "ifg-py"
repo_path = os.path.join(folder_path, project_name)

df_project = df[df["project"] == project_name].copy()
if df_project.empty:
    raise ValueError(f"Dataset tidak memiliki project {project_name!r}")

sample = df_project.iloc[2]
current_commit_hash = sample["commit_sha"]
parent_commit_hash = sample["parent_sha"]
repo = Repo(repo_path)
current_commit = repo.commit(current_commit_hash)
parent_commit = repo.commit(parent_commit_hash)
if current_commit.hexsha != str(current_commit_hash):
    raise ValueError("commit_sha tidak cocok dengan commit yang dibaca")
if parent_commit.hexsha != str(parent_commit_hash):
    raise ValueError("parent_sha tidak cocok dengan parent commit dataset")

diff = parent_commit.diff(current_commit, create_patch=True)
print("Current commit:", current_commit.hexsha)
print("Parent commit:", parent_commit.hexsha)
print("Dataset likely_bug:", sample["likely_bug"])
print("Changed entries:", len(diff))


NoSuchPathError: /home/darel/workspaces/tugas/rpl_ml/repositories/ifg-py

# Parse Python Files from Both Commits


In [9]:
def parse_commit_python_files(commit):
    trees = {}
    parse_errors = {}
    for item in commit.tree.traverse():
        if item.type != "blob" or not item.path.endswith(".py"):
            continue
        source = item.data_stream.read().decode("utf-8", errors="replace")
        try:
            trees[item.path] = py_to_ast(source)
        except ValueError as error:
            parse_errors[item.path] = str(error)
    return trees, parse_errors

trees_parent, parent_parse_errors = parse_commit_python_files(parent_commit)
trees_current, current_parse_errors = parse_commit_python_files(current_commit)
print("Parent Python files:", len(trees_parent))
print("Current Python files:", len(trees_current))
if parent_parse_errors or current_parse_errors:
    print("Parent parse errors:", parent_parse_errors)
    print("Current parse errors:", current_parse_errors)


NameError: name 'parent_commit' is not defined

# Build and Validate Full Repository Graphs


In [ ]:
builder_parent = Builder()
builder_parent.build(trees_parent)
builder_current = Builder()
builder_current.build(trees_current)

check_graph(builder_parent, trees_parent)
check_graph(builder_current, trees_current)


# Build Pair Training Inputs

Git diff is the source of truth for change ranges and edit records. `likely_bug` is only the optional target.

In [ ]:
diff_info = diff_metadata(diff)
if not diff_info["changed_files"]:
    raise ValueError("Tidak ada perubahan file Python pada commit pair ini")

parent_graph = prepare_pruned_graph(
    builder_parent, trees_parent, diff_info["old_ranges"], max_nodes=150
)
current_graph = prepare_pruned_graph(
    builder_current, trees_current, diff_info["new_ranges"], max_nodes=150
)

change_features = pair_features(diff_info)
parent_input = Tensor(
    parent_graph,
    graph_features=graph_features(parent_graph),
    local_change_features=change_features,
)
current_input = Tensor(
    current_graph,
    graph_features=graph_features(current_graph),
    local_change_features=change_features,
)

target = sample_target(sample.get("likely_bug"))

print("parent x:", parent_input.x.shape)
print("current x:", current_input.x.shape)
print("parent edge_index:", parent_input.edge_index.shape)
print("current edge_index:", current_input.edge_index.shape)
print("parent edge_type:", parent_input.edge_type.shape)
print("current edge_type:", current_input.edge_type.shape)
print("x differences:", np.count_nonzero(parent_input.x != current_input.x))
print("edge_index differences:", np.count_nonzero(parent_input.edge_index != current_input.edge_index))
print("edge_type differences:", np.count_nonzero(parent_input.edge_type != current_input.edge_type))
print("pair change features:", change_features)
print("likely_bug target:", target)

# Input Data Example

In [ ]:
# The label and change features belong to the commit pair, not to one graph.
# Keeping them at pair level prevents accidentally treating likely_bug as a node feature.
input_data = {
    "parent": {
        "x": parent_input.x,
        "edge_index": parent_input.edge_index,
        "edge_type": parent_input.edge_type,
        "graph_features": parent_input.graph_features,
    },
    "current": {
        "x": current_input.x,
        "edge_index": current_input.edge_index,
        "edge_type": current_input.edge_type,
        "graph_features": current_input.graph_features,
    },
    "change_features": change_features,
    "target": target,
}


def _describe_input(value):
    if isinstance(value, dict):
        return {key: _describe_input(item) for key, item in value.items()}
    return {
        "shape": tuple(value.shape) if hasattr(value, "shape") else (),
        "dtype": str(value.dtype) if hasattr(value, "dtype") else type(value).__name__,
    }

print("Input data prepared for the pair model:")
display(_describe_input(input_data))


# GNN

In [12]:
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F

def _as_tensor(value, dtype, device):
    return torch.as_tensor(value, dtype=dtype, device=device)

def validate_input_data(sample):
    required_graph_keys = {"x", "edge_index", "edge_type", "graph_features"}
    for version in ("parent", "current"):
        graph = sample[version]
        missing = required_graph_keys.difference(graph)
        if missing:
            raise ValueError(f"{version} graph is missing keys: {sorted(missing)}")
        x = np.asarray(graph["x"])
        edge_index = np.asarray(graph["edge_index"])
        edge_type = np.asarray(graph["edge_type"])
        if x.ndim != 2:
            raise ValueError(f"{version}.x must have shape [num_nodes, num_features]")
        if edge_index.ndim != 2 or edge_index.shape[0] != 2:
            raise ValueError(f"{version}.edge_index must have shape [2, num_edges]")
        if edge_index.shape[1] != edge_type.shape[0]:
            raise ValueError(f"{version}.edge_index and edge_type have different edge counts")
        if edge_index.size and (edge_index.min() < 0 or edge_index.max() >= x.shape[0]):
            raise ValueError(f"{version}.edge_index contains an invalid node index")
        if edge_type.size and (edge_type.min() < 0 or edge_type.max() >= len(EDGE_TYPES)):
            raise ValueError(f"{version}.edge_type contains an unknown relation id")
        if np.asarray(graph["graph_features"]).ndim != 1:
            raise ValueError(f"{version}.graph_features must be one-dimensional")
    if np.asarray(sample["change_features"]).ndim != 1:
        raise ValueError("change_features must be one-dimensional")
    target_value = float(np.asarray(sample["target"]).reshape(-1)[0])
    if target_value not in (0.0, 1.0):
        raise ValueError("target must be a binary value: 0 or 1")

class RelationalGCNLayer(nn.Module):
    def __init__(self, in_channels, out_channels, num_relations, dropout=0.0):
        super().__init__()
        if num_relations < 1:
            raise ValueError("num_relations must be positive")
        self.self_linear = nn.Linear(in_channels, out_channels, bias=False)
        self.relation_linears = nn.ModuleList(
            nn.Linear(in_channels, out_channels, bias = False)
            for _ in range(num_relations)
        )
        self.bias = nn.Parameter(torch.zeros(out_channels))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_type):
        num_nodes = x.shape[0]
        output = self.self_linear(x)
        source, destination = edge_index
        for relation_id, relation_linear in enumerate(self.relation_linears):
            mask = edge_type == relation_id
            if not torch.any(mask):
                continue
            relation_source = source[mask]
            relation_destination = destination[mask]
            messages = relation_linear(x[relation_source])
            degree = torch.zeros(num_nodes, device=x.device, dtype=x.dtype)
            degree.index_add_(0, relation_destination, torch.ones_like(relation_destination, dtype=x.dtype))
            normalizer = (degree[relation_destination] + 1.0).reciprocal().unsqueeze(1)
            output.index_add_(0, relation_destination, messages * normalizer)
        return self.dropout(output + self.bias)


class PairGCN(nn.Module):
    def __init__(self, node_features, graph_features, change_features, num_relations, hidden_channels=64, embedding_channels=64, dropout=0.2):
        super().__init__()
        self.encoder_1 = RelationalGCNLayer(node_features, hidden_channels, num_relations, dropout)
        self.encoder_2 = RelationalGCNLayer(hidden_channels, embedding_channels, num_relations, dropout)
        pair_features = embedding_channels * 8 + graph_features * 2 + change_features
        self.classifier = nn.Sequential(
            nn.Linear(pair_features, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, 1),
        )

    def encode_graph(self, graph):
        x = graph["x"]
        edge_index = graph["edge_index"]
        edge_type = graph["edge_type"]
        x = F.relu(self.encoder_1(x, edge_index, edge_type))
        x = F.relu(self.encoder_2(x, edge_index, edge_type))
        mean_pool = x.mean(dim=0)
        max_pool = x.max(dim=0).values
        return torch.cat((mean_pool, max_pool), dim=0)

    def forward(self, parent, current, change_features):
        parent_embedding = self.encode_graph(parent)
        current_embedding = self.encode_graph(current)
        pair_embedding = torch.cat(
            (
                parent_embedding,
                current_embedding,
                current_embedding - parent_embedding,
                current_embedding * parent_embedding,
                parent["graph_features"],
                current["graph_features"],
                change_features,
            ),
            dim=0,
        )
        return self.classifier(pair_embedding).reshape(())

def _tensorize_sample(sample, device):
    validate_input_data(sample)
    result = {}
    for version in ("parent", "current"):
        source = sample[version]
        result[version] = {
            "x": _as_tensor(source["x"], torch.float32, device),
            "edge_index": _as_tensor(source["edge_index"], torch.long, device),
            "edge_type": _as_tensor(source["edge_type"], torch.long, device),
            "graph_features": _as_tensor(source["graph_features"], torch.float32, device),
        }
    result["change_features"] = _as_tensor(sample["change_features"], torch.float32, device)
    result["target"] = _as_tensor(float(np.asarray(sample["target"]).reshape(-1)[0]), torch.float32, device)
    return result

def binary_metrics(logits, targets):
    predictions = (torch.sigmoid(torch.as_tensor(logits)) >= 0.5).long()
    targets = torch.as_tensor(targets).long()
    true_positive = int(((predictions == 1) & (targets == 1)).sum())
    true_negative = int(((predictions == 0) & (targets == 0)).sum())
    false_positive = int(((predictions == 1) & (targets == 0)).sum())
    false_negative = int(((predictions == 0) & (targets == 1)).sum())
    accuracy = (true_positive + true_negative) / max(len(targets), 1)
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    specificity = true_negative / max(true_negative + false_positive, 1)
    balanced_accuracy = (recall + specificity) / 2.0
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "balanced_accuracy": balanced_accuracy,
        "f1": f1,
        "true_positive": true_positive,
        "true_negative": true_negative,
        "false_positive": false_positive,
        "false_negative": false_negative,
    }

def evaluate_gnn(model, samples, criterion, device):
    model.eval()
    losses, logits, targets = [], [], []
    with torch.no_grad():
        for sample in samples:
            item = _tensorize_sample(sample, device)
            logit = model(item["parent"], item["current"], item["change_features"])
            losses.append(float(criterion(logit, item["target"])))
            logits.append(float(logit))
            targets.append(float(item["target"]))
    metrics = binary_metrics(logits, targets)
    metrics["loss"] = float(np.mean(losses)) if losses else float("nan")
    return metrics

def train_gnn(samples, epochs=30, learning_rate=1e-3, hidden_channels=64, embedding_channels=64, dropout=0.2, device=None):
    if not samples:
        raise ValueError("At least one input_data sample is required")
    labels = [float(np.asarray(sample["target"]).reshape(-1)[0]) for sample in samples]
    if len(set(labels)) < 2:
        warnings.warn(
            "Training data contains one class only; F1 and accuracy are not a meaningful validation estimate.",
            RuntimeWarning,
        )
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    prepared = [_tensorize_sample(sample, device) for sample in samples]
    node_features = prepared[0]["parent"]["x"].shape[1]
    graph_features = prepared[0]["parent"]["graph_features"].shape[0]
    change_features = prepared[0]["change_features"].shape[0]
    num_relations = len(EDGE_TYPES)
    model = PairGCN(
        node_features=node_features,
        graph_features=graph_features,
        change_features=change_features,
        num_relations=num_relations,
        hidden_channels=hidden_channels,
        embedding_channels=embedding_channels,
        dropout=dropout,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.BCEWithLogitsLoss()
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_losses, epoch_logits, epoch_targets = [], [], []
        for item in prepared:
            optimizer.zero_grad()
            logit = model(item["parent"], item["current"], item["change_features"])
            loss = criterion(logit, item["target"])
            loss.backward()
            optimizer.step()
            epoch_losses.append(float(loss.detach()))
            epoch_logits.append(float(logit.detach()))
            epoch_targets.append(float(item["target"]))
        metrics = binary_metrics(epoch_logits, epoch_targets)
        metrics["loss"] = float(np.mean(epoch_losses))
        metrics["epoch"] = epoch
        history.append(metrics)
    return model, history, evaluate_gnn(model, samples, criterion, device)


# One row is a smoke test only. A meaningful validation F1 requires multiple
# independent samples and a project/time-aware train-validation split.
model, history, training_metrics = train_gnn([input_data], epochs=30)
print("GNN training smoke test:")
print({key: round(value, 4) if isinstance(value, float) else value for key, value in training_metrics.items()})
print("final training loss:", round(history[-1]["loss"], 4))


KeyboardInterrupt: 

# Load Dataset

In [ ]:
dataset_path = "/home/darel/workspaces/tugas/rpl_ml/datasets/file-9.jsonl"
with open(dataset_path, "r", encoding="utf-8") as file:
    dataset = [json.loads(line) for line in file]
df = pd.DataFrame(dataset)
print(f"Dataset shape: {len(df)}")
display(df.head())

# Download Git

In [ ]:
import shutil
import subprocess

folder_path = "/home/darel/workspaces/tugas/rpl_ml/repositories"
for project in df["project"].unique():
    print("Project:", project)

    if os.path.exists(os.path.join(folder_path, project.split("/")[-1])):
        print("Repository already exists, skipping clone.")
        continue

    df_project = df[df["project"] == project]
    url_path = df_project["project_url"].iloc[0]
    repo_path = os.path.join(folder_path, project.split("/")[-1])

    try:
        subprocess.run(
            ["git", "clone", "--depth", "1", url_path, repo_path],
            timeout=60,
            check=True,
        )
    except subprocess.TimeoutExpired:
        print("Clone timeout setelah 1 menit, skip.")
        shutil.rmtree(repo_path, ignore_errors=True)
    except subprocess.CalledProcessError as error:
        print(f"Gagal clone repository, skip: {error}")
        shutil.rmtree(repo_path, ignore_errors=True)

In [18]:
train_data = []

for row in df.itertuples(index=False):
    project_name = row.project.split("/")[-1]
    repo_path = os.path.join(folder_path, project_name)
    if not os.path.exists(repo_path):
        # print(f"Repository {project_name} tidak ditemukan, skip.")
        continue
    try:
        sample = row._asdict()

        def normalize_sha(value):
            if isinstance(value, bytes):
                value = value.decode("utf-8")
            return str(value).strip()

        commit_sha = normalize_sha(row.commit_sha)
        parent_sha = normalize_sha(row.parent_sha)

        repo = Repo(repo_path)
        print(
            f"Processing project: {project_name}, "
            f"commit: {commit_sha}, parent: {parent_sha}"
        )

        current_commit = repo.commit(commit_sha)
        parent_commit = repo.commit(parent_sha)

        print(
            f"Project: {project_name}, "
            f"Commit: {current_commit.hexsha}, "
            f"Parent: {parent_commit.hexsha}"
        )

        if current_commit.hexsha != commit_sha:
            raise ValueError("commit_sha tidak cocok dengan commit yang dibaca")

        if parent_commit.hexsha != parent_sha:
            raise ValueError("parent_sha tidak cocok dengan parent commit dataset")

        diff = parent_commit.diff(current_commit, create_patch=True)
        print("Current commit:", current_commit.hexsha)
        print("Parent commit:", parent_commit.hexsha)
        print("Dataset likely_bug:", sample["likely_bug"])
        print("Changed entries:", len(diff))

        trees_parent, parent_parse_errors = parse_commit_python_files(parent_commit)
        trees_current, current_parse_errors = parse_commit_python_files(current_commit)
        print("Parent Python files:", len(trees_parent))
        print("Current Python files:", len(trees_current))
        if parent_parse_errors or current_parse_errors:
            print("Parent parse errors:", parent_parse_errors)
            print("Current parse errors:", current_parse_errors)

        builder_parent = Builder()
        builder_parent.build(trees_parent)
        builder_current = Builder()
        builder_current.build(trees_current)

        check_graph(builder_parent, trees_parent)
        check_graph(builder_current, trees_current)

        diff_info = diff_metadata(diff)
        if not diff_info["changed_files"]:
            raise ValueError("Tidak ada perubahan file Python pada commit pair ini")

        parent_graph = prepare_pruned_graph(
            builder_parent, trees_parent, diff_info["old_ranges"], max_nodes=150
        )
        current_graph = prepare_pruned_graph(
            builder_current, trees_current, diff_info["new_ranges"], max_nodes=150
        )

        change_features = pair_features(diff_info)
        parent_input = Tensor(
            parent_graph,
            graph_features=graph_features(parent_graph),
            local_change_features=change_features,
        )
        current_input = Tensor(
            current_graph,
            graph_features=graph_features(current_graph),
            local_change_features=change_features,
        )

        target = sample_target(sample.get("likely_bug"))

        print("parent x:", parent_input.x.shape)
        print("current x:", current_input.x.shape)
        print("parent edge_index:", parent_input.edge_index.shape)
        print("current edge_index:", current_input.edge_index.shape)
        print("parent edge_type:", parent_input.edge_type.shape)
        print("current edge_type:", current_input.edge_type.shape)
        print("x differences:", np.count_nonzero(parent_input.x != current_input.x))
        print("edge_index differences:", np.count_nonzero(parent_input.edge_index != current_input.edge_index))
        print("edge_type differences:", np.count_nonzero(parent_input.edge_type != current_input.edge_type))
        print("pair change features:", change_features)
        print("likely_bug target:", target)

        input_data = {
            "parent": {
                "x": parent_input.x,
                "edge_index": parent_input.edge_index,
                "edge_type": parent_input.edge_type,
                "graph_features": parent_input.graph_features,
            },
            "current": {
                "x": current_input.x,
                "edge_index": current_input.edge_index,
                "edge_type": current_input.edge_type,
                "graph_features": current_input.graph_features,
            },
            "change_features": change_features,
            "target": target,
        }

        train_data.append(input_data)
        
    except Exception as e:
            print(f"Gagal memproses commit untuk {project_name}: {e}")

print(f"Total samples prepared for training: {len(train_data)}")

model, history, training_metrics = train_gnn(train_data, epochs=30)
print("GNN training smoke test:")
print({key: round(value, 4) if isinstance(value, float) else value for key, value in training_metrics.items()})
print("final training loss:", round(history[-1]["loss"], 4))

Processing project: gajim, commit: 2e9c83914b3bf2150d578521538b3b86fe3dedd3, parent: e50b237ff2411bf0bee3240b79619bf5e3d88540
Gagal memproses commit untuk gajim: SHA b'2e9c83914b3bf2150d578521538b3b86fe3dedd3' could not be resolved, git returned: b'2e9c83914b3bf2150d578521538b3b86fe3dedd3 missing'
Processing project: gajim, commit: bc6f4289bef834e66a75e36ed2b20625d95b85cd, parent: 8c4288cc2c3316a33538348ddf34030a26d502a2
Gagal memproses commit untuk gajim: SHA b'bc6f4289bef834e66a75e36ed2b20625d95b85cd' could not be resolved, git returned: b'bc6f4289bef834e66a75e36ed2b20625d95b85cd missing'
Processing project: gajim, commit: 90a4fe3936ba28b683fe36737e0f8acaa841bed7, parent: 107cd477d03825cca5f097b26b3d968bb042768f
Gagal memproses commit untuk gajim: SHA b'90a4fe3936ba28b683fe36737e0f8acaa841bed7' could not be resolved, git returned: b'90a4fe3936ba28b683fe36737e0f8acaa841bed7 missing'
Processing project: gajim, commit: 44267b410c1697a316f08fea8b0cc542a7193aa0, parent: 1962321a2639ed0aa2

NameError: name 'train_gnn' is not defined